#### This notebook reads and filters DIA-NN main report parquet output - written for trap column testing


This notebook reads, filters, and retrieves simple numbers from the main report from DIANN output (report.parquet).
Results are exported as individual .csv files together with a summary .csv file.
Explanatory comments on the exported results are also exported as a .txt file (simple_numbers_df_explained.txt).




In [4]:
# import relevant packages
import sys
import os
import inspect
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from textwrap import dedent

# change working direction - point to where your notebooks (.ipynb) and python files (.py) are placed
new_cwd = Path("Z:/user/Dennis/trap_column_test/manuscript/diann_processing/jupyter_notebooks/github/")
os.chdir(new_cwd)
# verify new working directory
print("Current working directory:\n",os.getcwd())

# import custom functions
import diann_main_report_filtering as diann_filter

# auto reload of imported packages
%load_ext autoreload
%autoreload 2

# check where python is looking for path to resolve imports and where it actually loaded the packages from
print(f"This is where python is looking for packages:\n{sys.path}\n\n")

# change display settings for pandas dataframes
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

# set to back to default if needed
#pd.reset_option("display.max_columns")
#pd.reset_option("display.max_rows")
    
# check current working directory of this notebook
print("Current working directory:")
!pwd

Current working directory:
 Z:\user\Dennis\trap_column_test\manuscript\diann_processing\jupyter_notebooks\github
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
This is where python is looking for packages:
['C:\\Users\\dennis.dannecker\\AppData\\Local\\anaconda3\\envs\\trap_column_test\\python313.zip', 'C:\\Users\\dennis.dannecker\\AppData\\Local\\anaconda3\\envs\\trap_column_test\\DLLs', 'C:\\Users\\dennis.dannecker\\AppData\\Local\\anaconda3\\envs\\trap_column_test\\Lib', 'C:\\Users\\dennis.dannecker\\AppData\\Local\\anaconda3\\envs\\trap_column_test', '', 'C:\\Users\\dennis.dannecker\\AppData\\Local\\anaconda3\\envs\\trap_column_test\\Lib\\site-packages', 'C:\\Users\\dennis.dannecker\\AppData\\Local\\anaconda3\\envs\\trap_column_test\\Lib\\site-packages\\win32', 'C:\\Users\\dennis.dannecker\\AppData\\Local\\anaconda3\\envs\\trap_column_test\\Lib\\site-packages\\win32\\lib', 'C:\\Users\\dennis.dannecker\\AppData\\Local\\anaconda3\\envs\\trap_c

In [5]:
# define input and output directories for each acquired dataset

input_dir = Path("Z:/user/Dennis/trap_column_test/manuscript/diann_processing/dataset_a_combined/")
output_dir = Path("Z:/user/Dennis/trap_column_test/manuscript/diann_processing/dataset_a_combined/parquet_processing/github_test/")

In [6]:
# read in dianns main output parquet file as a pandas dataframe

# define input file name
input_file_name = "report.parquet"
# define input file path
input_file_path = input_dir / input_file_name
# read parquet file as pd dataframe
main_report_pd = pd.read_parquet(input_file_path, engine='pyarrow')
print("DIANN's parquet file was read in as a pandas dataframe.")

DIANN's parquet file was read in as a pandas dataframe.


In [ ]:
# split dataframe according to acquisition scheme so that each acquisition has its own dataframe

# ensure Run is string
main_report_pd["Run"] = main_report_pd["Run"].astype(str)
# split by exact Run value
run_dfs = {
    run_name: run_df.copy()
    for run_name, run_df in main_report_pd.groupby("Run")
}
# print the names of all acquired files and read in as a list
for name in run_dfs.keys():
    print(f"The following file was read in: {name}.")

In [5]:
# iterate through all dataframes and export them as separate csv files

for name in run_dfs.keys():
    run_dfs[name].to_csv(output_dir / (name + ".csv"), index=False)
    print(f"Exported file: {name} to .csv")

Exported file: 11512_10-11-2025_14-16-36_U_N2_DD_TH124_HeLa250pg_120SPD_pepmap_THIDtrap_test to .csv
Exported file: 11513_10-11-2025_14-27-12_U_N2_DD_TH124_HeLa250pg_120SPD_pepmap_THIDtrap_test to .csv
Exported file: 11514_10-11-2025_14-37-39_U_N2_DD_TH124_HeLa250pg_120SPD_pepmap_THIDtrap_test to .csv
Exported file: 11515_10-11-2025_14-48-13_U_N2_DD_TH124_HeLa250pg_120SPD_pepmap_THIDtrap_test to .csv
Exported file: 11516_10-11-2025_14-58-52_U_N2_DD_TH124_HeLa250pg_120SPD_pepmap_THIDtrap_test to .csv
Exported file: 11517_10-11-2025_15-09-26_U_N2_DD_TH124_HeLa250pg_120SPD_pepmap_THIDtrap_test to .csv
Exported file: 11518_10-11-2025_15-20-10_U_N2_DD_TH124_HeLa250pg_120SPD_pepmap_THIDtrap_test to .csv
Exported file: 11519_10-11-2025_15-30-37_U_N2_DD_TH124_HeLa250pg_120SPD_pepmap_THIDtrap_test to .csv
Exported file: 11520_10-11-2025_15-41-13_U_N2_DD_TH124_HeLa250pg_120SPD_pepmap_THIDtrap_test to .csv
Exported file: 11521_10-11-2025_15-51-58_U_N2_DD_TH124_HeLa250pg_120SPD_pepmap_THIDtrap_tes

In [6]:
# build main dataframe with aggregated numbers - proteins, peptides etc.

simple_numbers_rows = []
for run_name, df in run_dfs.items():
    # protein-group-level filtering
    df_prot_group_filt = diann_filter.filter_protein_group_confidence(
        df,
        protein_group_q_max=0.01,
        proteotypic_value=1
    )
    # protein-level filtering
    df_prot_filt = diann_filter.filter_protein_confidence(
        df,
        protein_q_max=0.01,
        proteotypic_value=1
    )
    # precursor-level filtering (Q.Value-based)
    df_prec_filt = diann_filter.filter_precursor_confidence(
        df,
        q_value_max=0.01,
        proteotypic_value=1
    )
    simple_numbers_rows.append({
        "Run": run_name,
        # protein-level counts
        "Unique.Protein.Groups": df_prot_group_filt["Protein.Group"].nunique(),
        "Unique.Proteins": df_prot_filt["Protein.Ids"].nunique(),
        # precursor-level counts
        "Unique.Precursor.Ids": df_prec_filt["Precursor.Id"].nunique(),
        "Unique.Modified.Sequences": df_prec_filt["Modified.Sequence"].nunique(),
        # average fwhm for all precursors (considers every precursors and distinguishes between differently charged precursors) in minutes
        "Average.FWHM.All.Precursors [min]": df_prec_filt["FWHM"].mean(),
        # average fwhm for all precursors (considers every precursors and distinguishes between differently charged precursors) in seconds
        "Average.FWHM.All.Precursors [sec]": df_prec_filt["FWHM"].mean()*60,
    })
# build final summary dataframe
simple_numbers_df = pd.DataFrame(simple_numbers_rows)
# documenting and explaining the exported simple numbers
simple_numbers_text = dedent("""\
The following explains the simple numbers stored in the "simple_numbers_df.csv" file.

- Unique.Protein.Groups: This is the number of unique protein groups identified by DIANN during the search.
  This number is reported at a Protein Group Q value of 0.01, i.e. it is filtered at 1% FDR at protein level (PG.Q.Value=0.01).
  Furthermore, only Protein Groups with corresponding proteotypic precursors are considered for counting (proteotypic_value=1).

- Unique.Proteins: This is the number of unique proteins identified by DIANN during the search.
  This number is reported at a Protein Q value of 0.01, i.e. it is filtered at 1% FDR at protein level (Protein.Q.Value=0.01).
  Furthermore, only Proteins with corresponding proteotypic precursors are considered for counting (proteotypic_value=1).

- Unique.Precursor.IDs: This is the number of unique precursors identified by DIANN during the search.
  It considers differently charged precursors as different and counts them separately.

  Example:
  1 AAAVLPVLDLAQR2  AAAVLPVLDLAQR
  2 AAAVLPVLDLAQR3  AAAVLPVLDLAQR
  --> Unique.Precursor.IDs counts two precursors.

  This number is reported at a Q value of 0.01, i.e. it is filtered at 1% FDR at precursor level (Q.Value=0.01).
  Furthermore, only proteotypic precursors are considered for counting (proteotypic_value=1).

- Unique.Modified.Sequences: This is the number of unique precursors identified by DIANN during the search.
  It considers differently charged precursors as the same and counts them only once.

  Example:
  1 AAAVLPVLDLAQR2  AAAVLPVLDLAQR
  2 AAAVLPVLDLAQR3  AAAVLPVLDLAQR
  --> Unique.Modified.Sequences counts one precursor.

  This number is reported at a Q value of 0.01, i.e. it is filtered at 1% FDR at precursor level (Q.Value=0.01).
  Furthermore, only proteotypic precursors are considered for counting (proteotypic_value=1).
  This number can be understood as the number of identified unique and proteotypic peptides.

- Average.FWHM.All.Precursors [min]: This is the average estimated full peak width at half-maximum.
  This can serve to evaluate chromatographic performance but only in direct comparison with very similar settings.
  A further caveat is that FWHM does not reflect any peak tailing.
  For forming the average, it considers the FWHM for every single identified precursor.

- Average.FWHM.All.Precursors [sec]: Same as [min] but multiplied by 60 to give seconds.
""")
                            

In [7]:
# define output file path

output_file = output_dir / "simple_numbers_df.csv"
output_file_txt = output_dir / "simple_numbers_df_explained.txt"
# export to CSV
simple_numbers_df.to_csv(output_file, index=False)
print("simple numbers were exported")
# save explaining text file
with open(output_file_txt, "w", encoding="utf-8") as f:
    f.write(simple_numbers_text)
# display simple number dataframe
simple_numbers_df

simple numbers were exported


,Run,Unique.Protein.Groups,Unique.Proteins,Unique.Precursor.Ids,Unique.Modified.Sequences,Average.FWHM.All.Precursors [min],Average.FWHM.All.Precursors [sec]
0,11512_10-11-2025_14-16-36_U_N2_DD_TH124_HeLa250pg_120SPD_pepmap_THIDtrap_test,3613,3617,25359,24039,0.029942,1.796503
1,11513_10-11-2025_14-27-12_U_N2_DD_TH124_HeLa250pg_120SPD_pepmap_THIDtrap_test,3559,3549,25120,23838,0.030258,1.815493
2,11514_10-11-2025_14-37-39_U_N2_DD_TH124_HeLa250pg_120SPD_pepmap_THIDtrap_test,3626,3619,25707,24314,0.030301,1.818036
3,11515_10-11-2025_14-48-13_U_N2_DD_TH124_HeLa250pg_120SPD_pepmap_THIDtrap_test,3570,3566,24976,23655,0.030077,1.804617
4,11516_10-11-2025_14-58-52_U_N2_DD_TH124_HeLa250pg_120SPD_pepmap_THIDtrap_test,3588,3579,25244,23864,0.030399,1.823921
...,...,...,...,...,...,...,...
75,11639_14-11-2025_02-52-51_U_N2_TH124_HeLa250pg_80SPD_new_pepmap_THIDtraptestd_,3622,3603,26204,24850,0.040357,2.421427
76,11640_14-11-2025_03-11-13_U_N2_TH124_HeLa250pg_80SPD_new_pepmap_THIDtraptestd_,3666,3653,26686,25205,0.039604,2.376259
77,11641_14-11-2025_03-29-34_U_N2_TH124_HeLa250pg_80SPD_new_pepmap_THIDtraptestd_,3651,3635,26536,25095,0.039911,2.394668
78,11642_14-11-2025_03-47-56_U_N2_TH124_HeLa250pg_80SPD_new_pepmap_THIDtraptestd_,3649,3653,26532,25114,0.040874,2.452413
